# Interactive dC/dt Timeseries

Browse processed NetCDF results and plot the Hutchinson–Mosier derivative, dC/dt.

Run the notebook from top to bottom, then:

1. Enter a results folder and click **Scan folder**.
2. Open the **Standard** or **MCMC** tab.
3. Select one or more compatible files and click **Load selected files**.
4. Use the timestamp and y-range sliders to focus the plot.
5. Keep **Filter outside y range** enabled to remove out-of-range points.
6. Enable **Moving-window mean** and choose a size in minutes, hours, or days to smooth by elapsed time.

Supported files:

- **Standard:** dimensions time × cutoff × deadband, without an MC dimension.
- **MCMC:** Pareto-selected dimensions time × MC, with dcdt(HM), best_deadband, and best_cutoff.

The file lists are intentionally left unselected after every scan. If selected files repeat timestamps, the notebook keeps the first non-null value in selected-file order and reports the merge in the tab status.

## Setup

In [1]:
import pathlib
import urllib.parse
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import ipywidgets as widgets
from IPython.display import clear_output, display

warnings.filterwarnings("ignore", category=FutureWarning)

CWD = pathlib.Path.cwd().resolve()
for candidate in [CWD, *CWD.parents]:
    if (candidate / "pyproject.toml").exists():
        PROJECT_ROOT = candidate
        break
else:
    PROJECT_ROOT = CWD

POST_PROCESSING_DIR = PROJECT_ROOT / "notebooks" / "post_processing"
DEFAULT_RESULTS_DIR = PROJECT_ROOT / "notebooks" / "processing" / "output"

## Data helpers

These helpers classify files from their contents, load selected files without Dask, merge overlapping timestamps deterministically, and reduce the datasets to plottable timeseries.

In [2]:
STANDARD_SCHEMA = "standard"
MCMC_SCHEMA = "mcmc_best_pareto"
REQUIRED_MCMC_VARIABLES = {"dcdt(HM)", "best_deadband", "best_cutoff"}


def sanitize_folder_path(path_value):
    text = str(path_value or "").strip()
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if lines:
        text = lines[0]
    if not text:
        raise FileNotFoundError("Results folder is empty.")

    if text.startswith(("Path(", "pathlib.Path(")) and text.endswith(")"):
        text = text[text.find("(") + 1:-1].strip()

    for _ in range(2):
        if len(text) >= 2 and text[0] == text[-1] and text[0] in {"'", '"'}:
            text = text[1:-1].strip()

    parsed = urllib.parse.urlparse(text)
    if parsed.scheme == "file":
        text = urllib.parse.unquote(parsed.path)
    if text.startswith("Users/"):
        text = "/" + text

    return pathlib.Path(text).expanduser().resolve()


def classify_result_dataset(ds):
    dims = set(ds.sizes)
    variables = set(ds.data_vars)

    if (
        {"time", "cutoff", "deadband"}.issubset(dims)
        and "MC" not in dims
        and "dcdt(HM)" in variables
    ):
        return STANDARD_SCHEMA

    if (
        {"time", "MC"}.issubset(dims)
        and "cutoff" not in dims
        and "deadband" not in dims
        and REQUIRED_MCMC_VARIABLES.issubset(variables)
    ):
        return MCMC_SCHEMA

    return "ignored"


def inspect_result_file(path):
    path = pathlib.Path(path)
    try:
        with xr.open_dataset(path) as ds:
            schema = classify_result_dataset(ds)
            return {
                "path": path,
                "file": path.name,
                "schema": schema,
                "dimensions": ", ".join(
                    f"{name}={size}" for name, size in ds.sizes.items()
                ),
                "variables": ", ".join(ds.data_vars),
                "error": "",
            }
    except Exception as exc:
        return {
            "path": path,
            "file": path.name,
            "schema": "unreadable",
            "dimensions": "",
            "variables": "",
            "error": f"{type(exc).__name__}: {exc}",
        }


def discover_result_files(folder_value):
    folder = sanitize_folder_path(folder_value)
    if not folder.exists():
        raise FileNotFoundError(f"Results folder does not exist: {folder}")
    if not folder.is_dir():
        raise NotADirectoryError(f"Results path is not a folder: {folder}")

    paths = sorted(path for path in folder.glob("*.nc") if path.is_file())
    if not paths:
        raise FileNotFoundError(f"No top-level .nc files found in {folder}")

    records = [inspect_result_file(path) for path in paths]
    standard = [record["path"] for record in records if record["schema"] == STANDARD_SCHEMA]
    mcmc = [record["path"] for record in records if record["schema"] == MCMC_SCHEMA]
    return folder, standard, mcmc, records


def _loaded_dataset(path):
    with xr.open_dataset(path) as opened:
        return opened.load()


def _format_timestamp_examples(index, limit=3):
    unique = pd.DatetimeIndex(index).unique().sort_values()
    return ", ".join(timestamp.isoformat() for timestamp in unique[:limit])


def combine_selected_files(files, expected_schema):
    paths = [pathlib.Path(path) for path in files]
    if not paths:
        raise ValueError("Select at least one compatible file.")

    datasets = []
    timestamp_parts = []
    for path in paths:
        ds = _loaded_dataset(path)
        actual_schema = classify_result_dataset(ds)
        if actual_schema != expected_schema:
            raise ValueError(
                f"{path.name} has schema {actual_schema!r}; "
                f"the selected tab requires {expected_schema!r}."
            )
        timestamps = pd.DatetimeIndex(pd.to_datetime(ds["time"].values))
        if timestamps.isna().any():
            raise ValueError(f"{path.name} contains invalid time values.")
        datasets.append(ds)
        timestamp_parts.append(timestamps)

    all_timestamps = timestamp_parts[0]
    for timestamps in timestamp_parts[1:]:
        all_timestamps = all_timestamps.append(timestamps)

    duplicate_mask = all_timestamps.duplicated(keep=False)
    duplicate_times = all_timestamps[duplicate_mask].unique().sort_values()
    overlap_count = len(duplicate_times)
    overlap_examples = _format_timestamp_examples(duplicate_times) if overlap_count else ""

    try:
        combined = xr.concat(
            datasets,
            dim="time",
            join="outer",
            data_vars="all",
            coords="minimal",
            compat="override",
        ).sortby("time")
        if overlap_count:
            # Preserve selected-file order and take the first non-null value per cell.
            # This also merges complementary cutoff/deadband grids safely.
            combined = combined.groupby("time").first(skipna=True).sortby("time")
    except Exception as exc:
        raise ValueError(
            "Selected files could not be combined. "
            "Choose files with compatible result schemas and coordinates."
        ) from exc

    if classify_result_dataset(combined) != expected_schema:
        raise ValueError("The combined dataset no longer matches the selected tab.")
    combined.attrs["overlap_count"] = int(overlap_count)
    combined.attrs["overlap_examples"] = overlap_examples
    return combined


def _python_scalar(value):
    return value.item() if hasattr(value, "item") else value


def valid_deadbands(ds):
    data = ds["dcdt(HM)"]
    availability = np.isfinite(data).any(dim=["time", "cutoff"])
    return [
        _python_scalar(value)
        for value, available in zip(ds["deadband"].values, availability.values)
        if bool(available)
    ]


def valid_cutoffs(ds, deadband):
    data = ds["dcdt(HM)"].sel(deadband=deadband)
    availability = np.isfinite(data).any(dim="time")
    return [
        _python_scalar(value)
        for value, available in zip(ds["cutoff"].values, availability.values)
        if bool(available)
    ]


def standard_series(ds, deadband, cutoff):
    data = ds["dcdt(HM)"].sel(deadband=deadband, cutoff=cutoff)
    if tuple(data.dims) != ("time",):
        data = data.transpose("time")
    series = pd.Series(
        np.asarray(data.values, dtype=float),
        index=pd.DatetimeIndex(pd.to_datetime(ds["time"].values), name="time"),
        name="dcdt(HM)",
    ).sort_index()
    if not np.isfinite(series.values).any():
        raise ValueError(
            f"No finite dcdt(HM) values exist for deadband={deadband}, cutoff={cutoff}."
        )
    return series


def summarize_mcmc(ds):
    dcdt = ds["dcdt(HM)"]
    if "MC" not in dcdt.dims or "time" not in dcdt.dims:
        raise ValueError("MCMC dcdt(HM) must have time and MC dimensions.")

    summary = pd.DataFrame(
        {
            "q16": np.asarray(dcdt.quantile(0.16, dim="MC", skipna=True).values, dtype=float),
            "median": np.asarray(dcdt.median(dim="MC", skipna=True).values, dtype=float),
            "q84": np.asarray(dcdt.quantile(0.84, dim="MC", skipna=True).values, dtype=float),
            "best_deadband": np.asarray(ds["best_deadband"].values),
            "best_cutoff": np.asarray(ds["best_cutoff"].values),
        },
        index=pd.DatetimeIndex(pd.to_datetime(ds["time"].values), name="time"),
    ).sort_index()

    finite_rows = np.isfinite(summary[["q16", "median", "q84"]]).all(axis=1)
    if not finite_rows.any():
        raise ValueError("The selected MCMC files contain no finite dcdt(HM) summaries.")
    ordered = summary.loc[finite_rows, ["q16", "median", "q84"]]
    if not ((ordered["q16"] <= ordered["median"]) & (ordered["median"] <= ordered["q84"])).all():
        raise ValueError("MCMC quantiles are not ordered as q16 ≤ median ≤ q84.")
    summary.attrs["overlap_count"] = int(ds.attrs.get("overlap_count", 0))
    summary.attrs["overlap_examples"] = ds.attrs.get("overlap_examples", "")
    return summary


def timestamp_slider_options(index):
    timestamps = pd.DatetimeIndex(index).dropna().unique().sort_values()
    if not len(timestamps):
        raise ValueError("No valid timestamps are available for the date slider.")
    return [
        (timestamp.strftime("%Y-%m-%d %H:%M:%S"), timestamp.to_datetime64())
        for timestamp in timestamps
    ]


def finite_plot_bounds(*values):
    finite_parts = []
    for value in values:
        array = np.asarray(value, dtype=float).ravel()
        finite_parts.append(array[np.isfinite(array)])
    finite = np.concatenate([part for part in finite_parts if part.size]) if any(
        part.size for part in finite_parts
    ) else np.array([], dtype=float)
    if not finite.size:
        raise ValueError("No finite values are available for the y-range slider.")

    lower = float(finite.min())
    upper = float(finite.max())
    if np.isclose(lower, upper):
        pad = max(abs(lower) * 0.10, 0.01)
    else:
        pad = 0.05 * (upper - lower)
    slider_min = lower - pad
    slider_max = upper + pad
    step = max((slider_max - slider_min) / 200.0, np.finfo(float).eps)
    return slider_min, slider_max, step


def crop_index(index, date_range):
    index = pd.DatetimeIndex(index)
    if not date_range:
        return np.ones(len(index), dtype=bool)
    start, end = (pd.Timestamp(value) for value in date_range)
    return (index >= start) & (index <= end)


def smoothing_offset(window_size, window_unit):
    size = int(window_size)
    if size < 1:
        raise ValueError("Moving-window size must be at least 1.")
    unit_aliases = {"minutes": "min", "hours": "h", "days": "d"}
    if window_unit not in unit_aliases:
        raise ValueError(f"Unsupported moving-window unit: {window_unit!r}")
    return pd.Timedelta(size, unit=unit_aliases[window_unit])


def filter_series_by_y_range(series, y_range, enabled=True):
    filtered = series.copy()
    if enabled and y_range:
        lower, upper = map(float, y_range)
        filtered = filtered.where(filtered.between(lower, upper, inclusive="both"))
    return filtered


def filter_mcmc_by_y_range(summary, y_range, enabled=True):
    filtered = summary.copy()
    if enabled and y_range:
        lower, upper = map(float, y_range)
        keep = filtered["median"].between(lower, upper, inclusive="both")
        filtered.loc[~keep, ["q16", "median", "q84"]] = np.nan
    return filtered


def smooth_series_by_time(series, enabled=False, window_size=60, window_unit="minutes"):
    if not enabled:
        return series.copy()
    offset = smoothing_offset(window_size, window_unit)
    return series.sort_index().rolling(offset, center=True, min_periods=1).mean()


def smooth_mcmc_by_time(summary, enabled=False, window_size=60, window_unit="minutes"):
    if not enabled:
        return summary.copy()
    offset = smoothing_offset(window_size, window_unit)
    smoothed = summary.copy()
    columns = ["q16", "median", "q84"]
    smoothed[columns] = summary[columns].sort_index().rolling(
        offset, center=True, min_periods=1
    ).mean()
    return smoothed


def count_outside_y_range(values, y_range):
    array = np.asarray(values, dtype=float)
    finite = np.isfinite(array)
    if not y_range:
        return 0
    lower, upper = map(float, y_range)
    return int((finite & ((array < lower) | (array > upper))).sum())


def overlap_note(source):
    attrs = getattr(source, "attrs", {})
    count = int(attrs.get("overlap_count", 0))
    if not count:
        return ""
    examples = attrs.get("overlap_examples", "")
    example_text = f" Examples: {examples}." if examples else ""
    return (
        f" <b>Overlap handled:</b> merged {count} repeated timestamp(s) "
        f"using the first non-null value in selected-file order.{example_text}"
    )

## Plot helpers

In [3]:
def plot_standard_timeseries(
    ds,
    deadband,
    cutoff,
    date_range=None,
    y_range=None,
    filter_y=True,
    smooth=False,
    window_size=60,
    window_unit="minutes",
    file_count=1,
):
    series = standard_series(ds, deadband, cutoff)
    visible = series.loc[crop_index(series.index, date_range)]
    visible = filter_series_by_y_range(visible, y_range, enabled=filter_y)
    visible = smooth_series_by_time(
        visible, enabled=smooth, window_size=window_size, window_unit=window_unit
    )
    visible = filter_series_by_y_range(visible, y_range, enabled=filter_y)
    finite = np.isfinite(visible.values)
    if not finite.any():
        raise ValueError(
            "No finite standard results remain inside the selected timestamp and y ranges."
        )

    line_label = (
        f"dcdt(HM), {window_size} {window_unit} moving mean"
        if smooth else "dcdt(HM)"
    )
    fig, ax = plt.subplots(figsize=(11.5, 4.2), dpi=120)
    ax.plot(
        visible.index,
        visible.values,
        color="#2b6cb0",
        marker="o",
        markersize=3,
        linewidth=1.0,
        label=line_label,
    )
    ax.axhline(0.0, color="0.35", linewidth=0.8, alpha=0.5)
    if y_range:
        ax.set_ylim(*y_range)
    ax.set_xlabel("Time")
    ax.set_ylabel("dC/dt (HM) [ppm s⁻¹]")
    ax.set_title(
        f"Standard dC/dt — deadband={deadband} s, cutoff={cutoff} s "
        f"— {file_count} file(s)"
    )
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
    fig.autofmt_xdate()
    fig.tight_layout()
    return fig


def plot_mcmc_timeseries(
    summary,
    date_range=None,
    y_range=None,
    filter_y=True,
    smooth=False,
    window_size=60,
    window_unit="minutes",
    file_count=1,
):
    mask = crop_index(summary.index, date_range)
    visible = summary.loc[mask]
    visible = filter_mcmc_by_y_range(visible, y_range, enabled=filter_y)
    visible = smooth_mcmc_by_time(
        visible, enabled=smooth, window_size=window_size, window_unit=window_unit
    )
    visible = filter_mcmc_by_y_range(visible, y_range, enabled=filter_y)
    finite = np.isfinite(visible[["q16", "median", "q84"]]).all(axis=1)
    if not finite.any():
        raise ValueError(
            "No finite MCMC results remain inside the selected timestamp and y ranges."
        )

    median_label = (
        f"posterior median, {window_size} {window_unit} moving mean"
        if smooth else "posterior median"
    )
    fig, ax = plt.subplots(figsize=(11.5, 4.2), dpi=120)
    ax.fill_between(
        visible.index,
        visible["q16"].values,
        visible["q84"].values,
        color="#2b6cb0",
        alpha=0.24,
        label="16–84%",
    )
    ax.plot(
        visible.index,
        visible["median"].values,
        color="#2b6cb0",
        marker="o",
        markersize=3,
        linewidth=1.1,
        label=median_label,
    )
    ax.axhline(0.0, color="0.35", linewidth=0.8, alpha=0.5)
    if y_range:
        ax.set_ylim(*y_range)
    ax.set_xlabel("Time")
    ax.set_ylabel("dC/dt (HM) [ppm s⁻¹]")
    ax.set_title(f"MCMC Pareto-selected dC/dt — {file_count} file(s)")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
    fig.autofmt_xdate()
    fig.tight_layout()
    return fig

## Interactive browser

The results-folder controls are shared. Each tab has its own explicit file selection, timestamp range, y-value filter, time-based moving-window smoothing controls, status, and plot output.

In [4]:
style = {"description_width": "145px"}
wide = widgets.Layout(width="900px")
path_layout = widgets.Layout(width="900px", height="72px")
file_layout = widgets.Layout(width="900px")
slider_layout = widgets.Layout(width="880px")

state = {
    "standard_ds": None,
    "standard_files": [],
    "mcmc_summary": None,
    "mcmc_files": [],
    "updating": False,
}

folder_widget = widgets.Textarea(
    value=str(DEFAULT_RESULTS_DIR),
    description="Results folder",
    placeholder="Paste a folder containing processed NetCDF result files.",
    continuous_update=False,
    style=style,
    layout=path_layout,
)
scan_button = widgets.Button(description="Scan folder", button_style="primary")
scan_status = widgets.HTML(value="Enter a results folder, then scan it.")
scan_output = widgets.Output(layout=wide)

standard_files_widget = widgets.SelectMultiple(
    options=[],
    value=(),
    description="Standard files",
    rows=9,
    style=style,
    layout=file_layout,
)
standard_load_button = widgets.Button(description="Load selected files", button_style="success")
standard_deadband_widget = widgets.Dropdown(options=[], description="Deadband [s]", disabled=True, style=style)
standard_cutoff_widget = widgets.Dropdown(options=[], description="Cutoff [s]", disabled=True, style=style)
standard_date_widget = widgets.SelectionRangeSlider(
    options=[("Load files first", 0)],
    index=(0, 0),
    description="Timestamp range",
    disabled=True,
    continuous_update=False,
    style=style,
    layout=slider_layout,
)
standard_y_widget = widgets.FloatRangeSlider(
    value=(0.0, 1.0),
    min=0.0,
    max=1.0,
    step=0.01,
    description="Y range",
    disabled=True,
    continuous_update=False,
    readout_format=".4g",
    style=style,
    layout=slider_layout,
)
standard_filter_y_widget = widgets.Checkbox(
    value=True, description="Filter outside y range", disabled=True, indent=False
)
standard_smooth_widget = widgets.Checkbox(
    value=False, description="Moving-window mean", disabled=True, indent=False
)
standard_window_size_widget = widgets.BoundedIntText(
    value=60, min=1, max=1_000_000, step=1, description="Window size",
    disabled=True, continuous_update=False, style=style,
)
standard_window_unit_widget = widgets.Dropdown(
    options=["minutes", "hours", "days"], value="minutes",
    description="Window unit", disabled=True, style=style,
)
standard_status = widgets.HTML(value="Scan a folder and select one or more standard files.")
standard_output = widgets.Output(layout=wide)

mcmc_files_widget = widgets.SelectMultiple(
    options=[],
    value=(),
    description="MCMC files",
    rows=9,
    style=style,
    layout=file_layout,
)
mcmc_load_button = widgets.Button(description="Load selected files", button_style="success")
mcmc_date_widget = widgets.SelectionRangeSlider(
    options=[("Load files first", 0)],
    index=(0, 0),
    description="Timestamp range",
    disabled=True,
    continuous_update=False,
    style=style,
    layout=slider_layout,
)
mcmc_y_widget = widgets.FloatRangeSlider(
    value=(0.0, 1.0),
    min=0.0,
    max=1.0,
    step=0.01,
    description="Y range",
    disabled=True,
    continuous_update=False,
    readout_format=".4g",
    style=style,
    layout=slider_layout,
)
mcmc_filter_y_widget = widgets.Checkbox(
    value=True, description="Filter outside y range", disabled=True, indent=False
)
mcmc_smooth_widget = widgets.Checkbox(
    value=False, description="Moving-window mean", disabled=True, indent=False
)
mcmc_window_size_widget = widgets.BoundedIntText(
    value=60, min=1, max=1_000_000, step=1, description="Window size",
    disabled=True, continuous_update=False, style=style,
)
mcmc_window_unit_widget = widgets.Dropdown(
    options=["minutes", "hours", "days"], value="minutes",
    description="Window unit", disabled=True, style=style,
)
mcmc_status = widgets.HTML(value="Scan a folder and select one or more best-Pareto MCMC files.")
mcmc_output = widgets.Output(layout=wide)


def _close_standard_dataset():
    if state["standard_ds"] is not None:
        try:
            state["standard_ds"].close()
        except Exception:
            pass
    state["standard_ds"] = None
    state["standard_files"] = []


def _disable_loaded_controls():
    state["updating"] = True
    try:
        _close_standard_dataset()
        state["mcmc_summary"] = None
        state["mcmc_files"] = []
        standard_deadband_widget.options = []
        standard_cutoff_widget.options = []
        standard_deadband_widget.disabled = True
        standard_cutoff_widget.disabled = True
        standard_date_widget.disabled = True
        standard_y_widget.disabled = True
        standard_filter_y_widget.disabled = True
        standard_smooth_widget.disabled = True
        standard_window_size_widget.disabled = True
        standard_window_unit_widget.disabled = True
        mcmc_date_widget.disabled = True
        mcmc_y_widget.disabled = True
        mcmc_filter_y_widget.disabled = True
        mcmc_smooth_widget.disabled = True
        mcmc_window_size_widget.disabled = True
        mcmc_window_unit_widget.disabled = True
    finally:
        state["updating"] = False

    standard_status.value = "Select one or more standard files, then load them."
    mcmc_status.value = "Select one or more best-Pareto MCMC files, then load them."
    with standard_output:
        clear_output(wait=True)
    with mcmc_output:
        clear_output(wait=True)


def _configure_date_slider(slider, index):
    options = timestamp_slider_options(index)
    slider.options = options
    slider.index = (0, len(options) - 1)
    slider.disabled = len(options) < 2


def _configure_y_slider(slider, *values):
    slider_min, slider_max, step = finite_plot_bounds(*values)
    # Expand first so positive-only or negative-only ranges never violate trait bounds.
    slider.min = min(slider.min, slider_min)
    slider.max = max(slider.max, slider_max)
    slider.step = step
    slider.value = (slider_min, slider_max)
    slider.min = slider_min
    slider.max = slider_max
    slider.disabled = False


def _display_figure(output_widget, fig):
    with output_widget:
        clear_output(wait=True)
        display(fig)
    plt.close(fig)


def _display_error(output_widget, status_widget, exc):
    status_widget.value = "<b>Error:</b> review the message below."
    with output_widget:
        clear_output(wait=True)
        print(f"{type(exc).__name__}: {exc}")


def _set_standard_cutoffs(preferred=None):
    ds = state["standard_ds"]
    if ds is None or standard_deadband_widget.value is None:
        standard_cutoff_widget.options = []
        standard_cutoff_widget.disabled = True
        return
    options = valid_cutoffs(ds, standard_deadband_widget.value)
    if not options:
        raise ValueError(f"No valid cutoffs exist for deadband={standard_deadband_widget.value}.")
    standard_cutoff_widget.options = options
    standard_cutoff_widget.value = preferred if preferred in options else options[0]
    standard_cutoff_widget.disabled = False


def _render_standard(reset_y=False):
    if state["updating"] or state["standard_ds"] is None:
        return
    try:
        deadband = standard_deadband_widget.value
        cutoff = standard_cutoff_widget.value
        if deadband is None or cutoff is None:
            return
        series = standard_series(state["standard_ds"], deadband, cutoff)

        if reset_y:
            state["updating"] = True
            try:
                _configure_y_slider(standard_y_widget, series.values)
            finally:
                state["updating"] = False

        fig = plot_standard_timeseries(
            state["standard_ds"],
            deadband=deadband,
            cutoff=cutoff,
            date_range=standard_date_widget.value,
            y_range=standard_y_widget.value,
            filter_y=standard_filter_y_widget.value,
            smooth=standard_smooth_widget.value,
            window_size=standard_window_size_widget.value,
            window_unit=standard_window_unit_widget.value,
            file_count=len(state["standard_files"]),
        )
        missing = int((~np.isfinite(series.values)).sum())
        suffix = f" {missing} timestamp(s) are unavailable for this window." if missing else ""
        selected_series = series.loc[crop_index(series.index, standard_date_widget.value)]
        filtered_count = (
            count_outside_y_range(selected_series.values, standard_y_widget.value)
            if standard_filter_y_widget.value else 0
        )
        filter_suffix = f" Filtered {filtered_count} raw point(s) outside the y range." if filtered_count else ""
        smooth_suffix = (
            f" Smoothed with a centered {standard_window_size_widget.value} "
            f"{standard_window_unit_widget.value} moving mean."
            if standard_smooth_widget.value else ""
        )
        standard_status.value = (
            f"<b>Loaded:</b> {len(series)} timestamps from "
            f"{len(state['standard_files'])} file(s).{suffix}{filter_suffix}"
            f"{smooth_suffix}{overlap_note(state['standard_ds'])}"
        )
        _display_figure(standard_output, fig)
    except Exception as exc:
        _display_error(standard_output, standard_status, exc)


def _render_mcmc():
    if state["updating"] or state["mcmc_summary"] is None:
        return
    try:
        fig = plot_mcmc_timeseries(
            state["mcmc_summary"],
            date_range=mcmc_date_widget.value,
            y_range=mcmc_y_widget.value,
            filter_y=mcmc_filter_y_widget.value,
            smooth=mcmc_smooth_widget.value,
            window_size=mcmc_window_size_widget.value,
            window_unit=mcmc_window_unit_widget.value,
            file_count=len(state["mcmc_files"]),
        )
        missing = int((~np.isfinite(state["mcmc_summary"][["q16", "median", "q84"]]).all(axis=1)).sum())
        suffix = f" {missing} timestamp(s) have incomplete posterior summaries." if missing else ""
        selected_summary = state["mcmc_summary"].loc[
            crop_index(state["mcmc_summary"].index, mcmc_date_widget.value)
        ]
        filtered_count = (
            count_outside_y_range(selected_summary["median"].values, mcmc_y_widget.value)
            if mcmc_filter_y_widget.value else 0
        )
        filter_suffix = f" Filtered {filtered_count} posterior median point(s) outside the y range." if filtered_count else ""
        smooth_suffix = (
            f" Smoothed with a centered {mcmc_window_size_widget.value} "
            f"{mcmc_window_unit_widget.value} moving mean."
            if mcmc_smooth_widget.value else ""
        )
        mcmc_status.value = (
            f"<b>Loaded:</b> {len(state['mcmc_summary'])} timestamps from "
            f"{len(state['mcmc_files'])} file(s).{suffix}{filter_suffix}"
            f"{smooth_suffix}{overlap_note(state['mcmc_summary'])}"
        )
        _display_figure(mcmc_output, fig)
    except Exception as exc:
        _display_error(mcmc_output, mcmc_status, exc)


def on_scan_clicked(_):
    with scan_output:
        clear_output(wait=True)
        try:
            folder, standard_files, mcmc_files, records = discover_result_files(folder_widget.value)
            _disable_loaded_controls()
            standard_files_widget.options = [(path.name, str(path)) for path in standard_files]
            standard_files_widget.value = ()
            mcmc_files_widget.options = [(path.name, str(path)) for path in mcmc_files]
            mcmc_files_widget.value = ()
            scan_status.value = (
                f"<b>Scanned:</b> {folder}<br>"
                f"Found {len(standard_files)} standard and "
                f"{len(mcmc_files)} best-Pareto MCMC file(s). "
                "Select files explicitly in either tab."
            )
            display(pd.DataFrame(records).drop(columns=["path"]))
        except Exception as exc:
            standard_files_widget.options = []
            standard_files_widget.value = ()
            mcmc_files_widget.options = []
            mcmc_files_widget.value = ()
            _disable_loaded_controls()
            scan_status.value = "<b>Scan failed.</b>"
            print(f"{type(exc).__name__}: {exc}")


def on_standard_load_clicked(_):
    try:
        selected = [pathlib.Path(value) for value in standard_files_widget.value]
        ds = combine_selected_files(selected, STANDARD_SCHEMA)
        deadbands = valid_deadbands(ds)
        if not deadbands:
            raise ValueError("Selected standard files contain no finite dcdt(HM) windows.")

        _close_standard_dataset()
        state["standard_ds"] = ds
        state["standard_files"] = selected

        state["updating"] = True
        try:
            standard_deadband_widget.options = deadbands
            standard_deadband_widget.value = deadbands[0]
            standard_deadband_widget.disabled = False
            _set_standard_cutoffs()
            _configure_date_slider(
                standard_date_widget,
                pd.DatetimeIndex(pd.to_datetime(ds["time"].values)),
            )
            series = standard_series(ds, standard_deadband_widget.value, standard_cutoff_widget.value)
            _configure_y_slider(standard_y_widget, series.values)
            standard_filter_y_widget.disabled = False
            standard_smooth_widget.disabled = False
            standard_window_size_widget.disabled = not standard_smooth_widget.value
            standard_window_unit_widget.disabled = not standard_smooth_widget.value
        finally:
            state["updating"] = False
        _render_standard()
    except Exception as exc:
        _display_error(standard_output, standard_status, exc)


def on_mcmc_load_clicked(_):
    try:
        selected = [pathlib.Path(value) for value in mcmc_files_widget.value]
        ds = combine_selected_files(selected, MCMC_SCHEMA)
        summary = summarize_mcmc(ds)
        ds.close()
        state["mcmc_summary"] = summary
        state["mcmc_files"] = selected

        state["updating"] = True
        try:
            _configure_date_slider(mcmc_date_widget, summary.index)
            _configure_y_slider(
                mcmc_y_widget,
                summary["q16"].values,
                summary["median"].values,
                summary["q84"].values,
            )
            mcmc_filter_y_widget.disabled = False
            mcmc_smooth_widget.disabled = False
            mcmc_window_size_widget.disabled = not mcmc_smooth_widget.value
            mcmc_window_unit_widget.disabled = not mcmc_smooth_widget.value
        finally:
            state["updating"] = False
        _render_mcmc()
    except Exception as exc:
        _display_error(mcmc_output, mcmc_status, exc)


def on_standard_deadband_changed(change):
    if state["updating"] or change.get("name") != "value":
        return
    try:
        state["updating"] = True
        try:
            _set_standard_cutoffs()
        finally:
            state["updating"] = False
        _render_standard(reset_y=True)
    except Exception as exc:
        _display_error(standard_output, standard_status, exc)


def on_standard_cutoff_changed(change):
    if not state["updating"] and change.get("name") == "value":
        _render_standard(reset_y=True)


def on_standard_view_changed(change):
    if not state["updating"] and change.get("name") == "value":
        _render_standard()


def on_standard_smoothing_changed(change):
    if state["updating"] or change.get("name") != "value":
        return
    standard_window_size_widget.disabled = not standard_smooth_widget.value
    standard_window_unit_widget.disabled = not standard_smooth_widget.value
    _render_standard()


def on_mcmc_view_changed(change):
    if not state["updating"] and change.get("name") == "value":
        _render_mcmc()


def on_mcmc_smoothing_changed(change):
    if state["updating"] or change.get("name") != "value":
        return
    mcmc_window_size_widget.disabled = not mcmc_smooth_widget.value
    mcmc_window_unit_widget.disabled = not mcmc_smooth_widget.value
    _render_mcmc()


scan_button.on_click(on_scan_clicked)
standard_load_button.on_click(on_standard_load_clicked)
mcmc_load_button.on_click(on_mcmc_load_clicked)
standard_deadband_widget.observe(on_standard_deadband_changed, names="value")
standard_cutoff_widget.observe(on_standard_cutoff_changed, names="value")
standard_date_widget.observe(on_standard_view_changed, names="value")
standard_y_widget.observe(on_standard_view_changed, names="value")
standard_filter_y_widget.observe(on_standard_view_changed, names="value")
standard_smooth_widget.observe(on_standard_smoothing_changed, names="value")
standard_window_size_widget.observe(on_standard_view_changed, names="value")
standard_window_unit_widget.observe(on_standard_view_changed, names="value")
mcmc_date_widget.observe(on_mcmc_view_changed, names="value")
mcmc_y_widget.observe(on_mcmc_view_changed, names="value")
mcmc_filter_y_widget.observe(on_mcmc_view_changed, names="value")
mcmc_smooth_widget.observe(on_mcmc_smoothing_changed, names="value")
mcmc_window_size_widget.observe(on_mcmc_view_changed, names="value")
mcmc_window_unit_widget.observe(on_mcmc_view_changed, names="value")

folder_controls = widgets.VBox([folder_widget, widgets.HBox([scan_button]), scan_status, scan_output])
standard_tab = widgets.VBox([
    widgets.HTML("<b>Standard results</b><br>Select non-MCMC grid files, then choose a deadband and cutoff."),
    standard_files_widget,
    standard_load_button,
    widgets.HBox([standard_deadband_widget, standard_cutoff_widget]),
    standard_date_widget,
    standard_y_widget,
    widgets.HBox([standard_filter_y_widget, standard_smooth_widget]),
    widgets.HBox([standard_window_size_widget, standard_window_unit_widget]),
    standard_status,
    standard_output,
])
mcmc_tab = widgets.VBox([
    widgets.HTML("<b>MCMC results</b><br>Select Pareto-chosen time × MC files to plot posterior summaries."),
    mcmc_files_widget,
    mcmc_load_button,
    mcmc_date_widget,
    mcmc_y_widget,
    widgets.HBox([mcmc_filter_y_widget, mcmc_smooth_widget]),
    widgets.HBox([mcmc_window_size_widget, mcmc_window_unit_widget]),
    mcmc_status,
    mcmc_output,
])

tabs = widgets.Tab(children=[standard_tab, mcmc_tab])
tabs.set_title(0, "Standard")
tabs.set_title(1, "MCMC")
display(widgets.VBox([folder_controls, tabs]))